# 7.2.2 Dataset と DataLoader の今の書き方

`7-2_torchtext.ipynb` は `torchtext.data.Field` で Dataset と DataLoader を作る。このファイルは、同じデータと前処理を、今の PyTorch の `Dataset` と `DataLoader` で書き直したものである。

`torchtext.data.Field` は torchtext 0.9 で `torchtext.legacy` に移り、0.12 で削除された。torchtext 自体も、その後メンテナンスが止まっている。文章とラベルをミニバッチにする処理は `torch.utils.data` に書く。

7-5 以降のノートブックは、まだ 7-2 の `Field` と `TEXT.vocab` を前提にしている。このファイルは 7-2 の読み替え用で、後ろの章はそのままにする。

# 7.2.2 学習目標

1.	7-2 の `Field`、`TabularDataset`、`Iterator` を、`Dataset` と `DataLoader` で実装できる

# 事前準備

- データは 7-2 と同じ `data/text_train.tsv`、`data/text_val.tsv`、`data/text_test.tsv` を使う
- 単語分割も 7-2 と同じ Janome を使う

# 1. 前処理と単語分割の関数を実装

この節は 7-2 の「1. 前処理と単語分割の関数を実装」と同じである。`Field` の `tokenize` に渡していた関数を、ここでは `Dataset` から呼ぶ。

In [8]:
# 単語分割にはJanomeを使用
from janome.tokenizer import Tokenizer

j_t = Tokenizer()


def tokenizer_janome(text):
    return [tok for tok in j_t.tokenize(text, wakati=True)]

In [9]:
# 前処理として正規化をする関数を定義
import re


def preprocessing_text(text):
    # 半角・全角の統一
    # 今回は無視

    # 英語の小文字化
    # 今回はここでは無視
    # output = output.lower()

    # 改行、半角スペース、全角スペースを削除
    text = re.sub('\r', '', text)
    text = re.sub('\n', '', text)
    text = re.sub('　', '', text)
    text = re.sub(' ', '', text)

    # 数字文字の一律「0」化
    text = re.sub(r'[0-9 ０-９]', '0', text)  # 数字

    # 記号と数字の除去
    # 今回は無視。半角記号,数字,英字
    # 今回は無視。全角記号

    # 特定文字を正規表現で置換する
    # 今回は無視

    return text

In [10]:
# 前処理とJanomeの単語分割を合わせた関数を定義する


def tokenizer_with_preprocessing(text):
    text = preprocessing_text(text)  # 前処理の正規化
    ret = tokenizer_janome(text)  # Janomeの単語分割

    return ret


# 動作確認
text = "昨日は とても暑く、気温が36度もあった。"
print(tokenizer_with_preprocessing(text))

['昨日', 'は', 'とても', '暑く', '、', '気温', 'が', '00', '度', 'も', 'あっ', 'た', '。']


# 2. 文章データの読み込み

7-2 では、読み込んだ文章とラベルに対する処理を `Field` で定義し、`TabularDataset` で tsv を読んでいた。

```python
max_length = 25
TEXT = torchtext.data.Field(sequential=True, tokenize=tokenizer_with_preprocessing,
                            use_vocab=True, lower=True, include_lengths=True, batch_first=True, fix_length=max_length)
LABEL = torchtext.data.Field(sequential=False, use_vocab=False)

train_ds, val_ds, test_ds = torchtext.data.TabularDataset.splits(
    path='./data/',
    train='text_train.tsv',
    validation='text_val.tsv',
    test='text_test.tsv',
    format='tsv',
    fields=[('Text', TEXT), ('Label', LABEL)])
```

`Field` の引数は、次の場所に分ける。

| 7-2 の引数 | このファイルでの置き場所 |
| --- | --- |
| `tokenize` | `Dataset` を作るときに `tokenizer_with_preprocessing` を呼ぶ |
| `lower=True` | 単語分割の前に `text.lower()` する |
| `use_vocab=True` | 次の節で単語を ID にする。ラベルは 7-2 の `use_vocab=False` と同じく数値のまま持つ |
| `include_lengths=True` | `DataLoader` に渡す関数で、パディング前の単語数も返す |
| `batch_first=True` と `fix_length` | 同じ関数で、長さ 25 の `[バッチ, 25]` に揃える |

`sequential` は「文章は長さが可変、ラベルは1個」という意味だった。ここでは文章をリスト、ラベルを1個の値として持てば足りる。

In [11]:
from torch.utils.data import Dataset


class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。7-2 の TabularDataset に相当する。"""

    def __init__(self, path, tokenizer):
        self.samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label = line.rstrip("\n").split("\t")
                # Field の lower=True と同じく、分割の前に小文字化する
                tokens = tokenizer(text.lower())
                self.samples.append((tokens, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


train_ds = TextDataset("./data/text_train.tsv", tokenizer_with_preprocessing)
val_ds = TextDataset("./data/text_val.tsv", tokenizer_with_preprocessing)
test_ds = TextDataset("./data/text_test.tsv", tokenizer_with_preprocessing)

# 動作確認。7-2 の vars(train_ds[0]) と同じ中身になる
print("訓練データの数", len(train_ds))
print("1つ目の訓練データ", {"Text": train_ds[0][0], "Label": train_ds[0][1]})
print("2つ目の訓練データ", {"Text": train_ds[1][0], "Label": train_ds[1][1]})

訓練データの数 4
1つ目の訓練データ {'Text': ['王', 'と', '王子', 'と', '女王', 'と', '姫', 'と', '男性', 'と', '女性', 'が', 'い', 'まし', 'た', '。'], 'Label': '0'}
2つ目の訓練データ {'Text': ['機械', '学習', 'が', '好き', 'です', '。'], 'Label': '1'}


# 単語の数値化

7-2 では `TEXT.build_vocab(train_ds, min_freq=1)` のあと、`TEXT.vocab.freqs` と `TEXT.vocab.stoi` を見ていた。

ここでは訓練データの単語から辞書を作る。`min_freq` 未満の単語は入れない。ID の並びは 7-2 に合わせる。`<unk>` が 0、`<pad>` が 1、その後は出現回数が多い順、回数が同じなら文字コード順である。

In [12]:
from collections import Counter


def build_vocab(dataset, min_freq=1):
    """7-2 の TEXT.build_vocab に相当する。"""
    freqs = Counter()
    for tokens, _ in dataset:
        freqs.update(tokens)

    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))

    itos = ["<unk>", "<pad>"] + words
    stoi = {word: index for index, word in enumerate(itos)}
    return freqs, stoi, itos


freqs, stoi, itos = build_vocab(train_ds, min_freq=1)

# 訓練データ内の単語と頻度を出力する。7-2 の TEXT.vocab.freqs に相当する
freqs

Counter({'と': 5,
         '。': 4,
         'の': 4,
         '文章': 4,
         'な': 4,
         'が': 3,
         '、': 3,
         'を': 3,
         'し': 3,
         '本章': 2,
         'ます': 2,
         '評価': 2,
         'て': 2,
         'いる': 2,
         'か': 2,
         '分類': 2,
         '王': 1,
         '王子': 1,
         '女王': 1,
         '姫': 1,
         '男性': 1,
         '女性': 1,
         'い': 1,
         'まし': 1,
         'た': 1,
         '機械': 1,
         '学習': 1,
         '好き': 1,
         'です': 1,
         'から': 1,
         '自然': 1,
         '言語': 1,
         '処理': 1,
         'に': 1,
         '取り組み': 1,
         'で': 1,
         'は': 1,
         '商品': 1,
         'レビュー': 1,
         '短い': 1,
         'に対して': 1,
         'その': 1,
         'ネガティブ': 1,
         'ポジティブ': 1,
         '0': 1,
         '値': 1,
         'クラス': 1,
         'する': 1,
         'モデル': 1,
         '構築': 1})

In [13]:
# 単語を id に変換する辞書。7-2 の TEXT.vocab.stoi に相当する
# 辞書に無い単語は <unk> の 0 になる
stoi

{'<unk>': 0,
 '<pad>': 1,
 'と': 2,
 '。': 3,
 'な': 4,
 'の': 5,
 '文章': 6,
 '、': 7,
 'が': 8,
 'し': 9,
 'を': 10,
 'いる': 11,
 'か': 12,
 'て': 13,
 'ます': 14,
 '分類': 15,
 '本章': 16,
 '評価': 17,
 '0': 18,
 'い': 19,
 'から': 20,
 'する': 21,
 'その': 22,
 'た': 23,
 'で': 24,
 'です': 25,
 'に': 26,
 'に対して': 27,
 'は': 28,
 'まし': 29,
 'クラス': 30,
 'ネガティブ': 31,
 'ポジティブ': 32,
 'モデル': 33,
 'レビュー': 34,
 '値': 35,
 '処理': 36,
 '取り組み': 37,
 '商品': 38,
 '女性': 39,
 '女王': 40,
 '好き': 41,
 '姫': 42,
 '学習': 43,
 '構築': 44,
 '機械': 45,
 '王': 46,
 '王子': 47,
 '男性': 48,
 '短い': 49,
 '自然': 50,
 '言語': 51}

# DataLoaderの作成

7-2 では `torchtext.data.Iterator` がミニバッチを作っていた。検証データは `train=False, sort=False` なので、ファイルに書いてある順のまま取り出していた。

```python
train_dl = torchtext.data.Iterator(train_ds, batch_size=2, train=True)
val_dl = torchtext.data.Iterator(val_ds, batch_size=2, train=False, sort=False)
test_dl = torchtext.data.Iterator(test_ds, batch_size=2, train=False, sort=False)

batch = next(iter(val_dl))
print(batch.Text)
print(batch.Label)
```

`batch.Text` は `(単語IDのテンソル, 各文の単語数)` だった。`include_lengths=True` のためである。`batch_first=True` と `fix_length=25` のため、テンソルの形は `[バッチ, 25]` になる。短い文は `<pad>` で埋め、長い文は 25 単語で切る。

その整形を `collate_batch` に書く。`DataLoader` はバッチを取り出すときにこの関数を呼ぶ。

In [14]:
import torch
from torch.utils.data import DataLoader

max_length = 25
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]


def collate_batch(batch):
    """7-2 の include_lengths、batch_first、fix_length に相当する。"""
    text_batch = []
    lengths = []
    labels = []

    for tokens, label in batch:
        ids = [stoi.get(token, unk_id) for token in tokens]
        length = min(len(ids), max_length)
        ids = ids[:max_length]
        ids += [pad_id] * (max_length - len(ids))

        text_batch.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(length)
        labels.append(int(label))

    text = torch.stack(text_batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return text, lengths, labels


# train=True はシャッフルする、という意味だった
train_dl = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_batch)
val_dl = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_ds, batch_size=2, shuffle=False, collate_fn=collate_batch)

# 動作確認。7-2 の batch.Text と batch.Label に相当する
text, lengths, labels = next(iter(val_dl))
print((text, lengths))
print(labels)

(tensor([[46,  2, 47,  2, 40,  2, 42,  2, 48,  2, 39,  8, 19, 29, 23,  3,  1,  1,
          1,  1,  1,  1,  1,  1,  1],
        [45, 43,  8, 41, 25,  3,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  1,  1,  1,  1,  1]]), tensor([16,  6]))
tensor([0, 1])


以上

検証データの最初のバッチは、7-2 の `batch.Text` と `batch.Label` と同じ並びになる。単語 ID は `stoi` の作り方を 7-2 に合わせてある。

In [15]:
batch = next(iter(val_dl))
batch

(tensor([[46,  2, 47,  2, 40,  2, 42,  2, 48,  2, 39,  8, 19, 29, 23,  3,  1,  1,
           1,  1,  1,  1,  1,  1,  1],
         [45, 43,  8, 41, 25,  3,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
           1,  1,  1,  1,  1,  1,  1]]),
 tensor([16,  6]),
 tensor([0, 1]))